# FEM Model: Dynamic Analysis of Pendulum Head Wall Impact


In [36]:
from netgen.occ import *
from netgen.geom2d import SplineGeometry
from ngsolve import *
from ngsolve.solvers import *
from ngsolve.webgui import Draw
from netgen.webgui import Draw as DrawGeo

import numpy as np

In [37]:
from ngsolve.timestepping import Newmark
from ngsolve.internal import VideoStart

## 2D Pendulum Model without Impact

In [107]:
# ---------- Geometry ----------
L  = 1.0      # rod length
h  = 0.08     # rod height
R  = 0.12     # head radius
rH = 0.01     # hinge (small) radius to clamp
ms = 0.03     # mesh size hint

In [ ]:
rod = MoveTo(0,-h/2).Rectangle(L, h).Face()
head = MoveTo(L,0).Circle(R).Face()

hinge = MoveTo(0.05,0).Circle(rH).Face()

geo = rod + head - hinge

geo.

DrawGeo(geo)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': …

BaseWebGuiScene

In [109]:
mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=0.025))
mesh.Curve(2)
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [110]:
mesh.GetBoundaries()

('default', 'default', 'default', 'default', 'default')

In [83]:
# ---------- Material / model parameters ----------
E   = 2.0e11       # Young's modulus [Pa]
nu  = 0.3          # Poisson ratio [-]
rho = 7800.0       # density [kg/m^3]
gacc = 9.81        # gravity [m/s^2]
thick = 1.0        # out-of-plane thickness for plane-stress-like mass [m]

# Lame parameters
mu  = E/(2*(1+nu))
lam = E*nu/((1+nu)*(1-2*nu))

In [84]:
# ---------- FE space ----------
fes = VectorH1(mesh, order=2, dim=3, dirichlet="hinge")   # pin at small circle, 3D vectors
u  = fes.TrialFunction()
v  = fes.TestFunction()

def Strain(w):     return Sym(Grad(w))
def Stress(w):     return 2*mu*Strain(w) + lam*Trace(Strain(w))*Id(3)  # Use Id(3) for 3D

# Stiffness (elasticity)
a = BilinearForm(fes, symmetric=True)
a += InnerProduct(Stress(u), Strain(v))*dx

# Mass (lumped/thick)
m = BilinearForm(fes, symmetric=True)
m += rho*thick*InnerProduct(u, v)*dx

# Gravity load
f = LinearForm(fes)
f += rho*thick*InnerProduct(CF((0, -gacc, 0)), v)*dx

a.Assemble()
m.Assemble()
f.Assemble()

used dof inconsistency
(silence this warning by setting BilinearForm(...check_unused=False) )
used dof inconsistency
(silence this warning by setting BilinearForm(...check_unused=False) )


In [85]:
import math
# ---------- Initial conditions ----------
# initial rigid-body rotation around hinge by q0 (45°)
q0 = math.radians(45.0)
c, s = math.cos(q0), math.sin(q0)
x = x
y = y
# R*x - x (rigid rotation minus identity)
u0x = (c-1.0)*x - s*y
u0y = s*x + (c-1.0)*y

u_gf  = GridFunction(fes, name="u")
v_gf  = GridFunction(fes, name="v")
acc_gf = GridFunction(fes, name="acc")

# Set initial displacement as a vector-valued coefficient function
u_gf.Set((u0x, u0y, 0), definedon=mesh.Materials("1"))
v_gf.Set((0,0,0))

In [86]:
# Compute consistent initial acceleration from M a0 = f - K u0
rhs0 = f.vec.CreateVector()
a.mat.Mult(u_gf.vec, rhs0)
rhs0.data = f.vec - rhs0
free = fes.FreeDofs()
Minv = m.mat.Inverse(free, inverse="sparsecholesky")
Minv.Mult(rhs0, acc_gf.vec)

In [89]:
# ---------- Time integration (Newmark) ----------
dt   = 5e-4
tend = 1.2
beta = 0.25
gamma = 0.5

# Effective matrix: M + beta*dt^2*K
Keff = m.mat.CreateMatrix()
Keff.AsVector().data = m.mat.AsVector()
Keff.AsVector().data += (beta*dt*dt) * a.mat.AsVector()
KeffInv = Keff.Inverse(free, inverse="sparsecholesky")

scene = Draw(u_gf, mesh, deformation=u_gf, order=2, draw_surf=False)
# (Optional) scale the visual deformation if the geometry is very stiff:
scene.deformscale = 1.0

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [88]:
t = 0.0
with TaskManager():
    while t < tend:
        # Predict displacement
        u_pred = u_gf.vec.CreateVector()
        u_pred.data = u_gf.vec + dt*v_gf.vec + dt*dt*(0.5-beta)*acc_gf.vec

        # RHS for a_{n+1}
        rhs = f.vec.CreateVector()
        temp = rhs.CreateVector()
        a.mat.Mult(u_pred, temp)
        rhs.data = f.vec - temp

        # Solve for a_{n+1}
        a_new = rhs.CreateVector()
        KeffInv.Mult(rhs, a_new)

        # Update v, u
        v_gf.vec.data = v_gf.vec + dt*((1.0 - gamma)*acc_gf.vec + gamma*a_new)
        u_gf.vec.data = u_pred + (beta*dt*dt) * a_new

        # Roll
        acc_gf.vec.data = a_new

        t += dt
        if int(t/dt) % 10 == 0:
            scene.Redraw()

print("Done.")

Done.


## 2D Ball falling on a Wall

In [7]:
# Parameters
y_pos_ball = 0.25
radius_ball = 0.1

length_wall = 0.4
thickness_wall = 0.05

ball = Circle((0, y_pos_ball), radius_ball).Face()
ball.edges.name = "ball"

wall = Rectangle(length_wall, thickness_wall).Face().Move((-length_wall/2, 0, 0))
wall.edges.Max(Y).name = "wall_contact"
wall.edges.Min(Y).name = "wall_fixed"
wall.edges.Min(X).name = "wall_fixed"
wall.edges.Max(X).name = "wall_fixed"

geo = Compound([ball, wall])

DrawGeo(geo)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': …

BaseWebGuiScene

In [8]:
mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=0.025))
mesh.Curve(2)
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [9]:
mesh.GetBoundaries()

('ball', 'wall_fixed', 'wall_fixed', 'wall_contact', 'wall_fixed')

In [10]:
g = 9.81
E, nu = 210e6, 0.2
rho = 7e3
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

In [11]:
I = Id(mesh.dim)

def C(u): 
    F = I+Grad(u)
    return F.trans*F
def NeoHooke (C):
    return 0.5 * mu * (Trace(C-I) + 2*mu/lam * Det(C)**(-lam/2/mu) - 1)

In [13]:
fes = VectorH1(mesh, order=3, dirichlet="wall_fixed")
u,v = fes.TnT()

In [14]:
force = CF((0,-g*rho))

In [15]:
uold = GridFunction(fes)
unew = GridFunction(fes)
vel = GridFunction(fes)
anew = GridFunction(fes)
aold = GridFunction(fes)

In [16]:
tau = 5e-3
tend = 1

bfmstar = BilinearForm(fes)
bfmstar += Variation( NeoHooke (C(u)).Compile(False)*dx )
bfmstar += Variation( -force*u*dx )
bfmstar += Variation( rho/2* 2/tau**2 * (u-uold-tau*vel-tau**2/4*aold)**2 * dx )

In [17]:
scene = Draw (unew, mesh, "disp", deformation=unew);

t = 0
unew.Set( (0,0) )
vel.Set( (0,0) )
contact = ContactBoundary(mesh.Boundaries("wall_contact|ball"), mesh.Boundaries("wall_contact|ball"))

X = CoefficientFunction((x,y))

if True:
    cf = (X + u-uold - (X.Other() + u.Other() - uold.Other())) * (-specialcf.normal(2).Other())
    # cf = (X + u-uold)*specialcf.normal(2) + \
    #    (X.Other() + u.Other() - uold.Other()) * specialcf.normal(2).Other()
    contact.AddEnergy(IfPos(cf, 1e9*cf*cf, 0), deformed=True)
else:
    cf = (X + u - (X.Other() + u.Other())) * contact.normal
    contact.AddEnergy(IfPos(cf, 1e9*cf*cf, 0), deformed=False)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [18]:
with TaskManager():
    while t < tend:
        print ("time", t)
        t += tau
        uold.vec.data = unew.vec
        aold.vec.data = anew.vec

        contact.Update(uold, bfmstar, 5, 0.01)
        NewtonMinimization (a=bfmstar, u=unew, printing=False, inverse="sparsecholesky")

        anew.vec.data = unew.vec-uold.vec-tau*vel.vec-tau**2/4*aold.vec
        anew.vec.data *= 4/tau**2
        vel.vec.data += 0.5*tau*aold.vec
        vel.vec.data += 0.5*tau*anew.vec

        # Redraw every 10 time steps
        if int(t/tau) % 5 == 0:
            scene.Redraw()

time 0
time 0.005
time 0.01
time 0.015
time 0.02
time 0.025
time 0.030000000000000002
time 0.035
time 0.04
time 0.045
time 0.049999999999999996
time 0.05499999999999999
time 0.05999999999999999
time 0.06499999999999999
time 0.06999999999999999
time 0.075
time 0.08
time 0.085
time 0.09000000000000001
time 0.09500000000000001
time 0.10000000000000002
time 0.10500000000000002
time 0.11000000000000003
time 0.11500000000000003
time 0.12000000000000004
time 0.12500000000000003
time 0.13000000000000003
time 0.13500000000000004
time 0.14000000000000004
time 0.14500000000000005
time 0.15000000000000005
time 0.15500000000000005
time 0.16000000000000006
time 0.16500000000000006
time 0.17000000000000007
time 0.17500000000000007
time 0.18000000000000008
time 0.18500000000000008
time 0.19000000000000009
time 0.1950000000000001
time 0.2000000000000001
time 0.2050000000000001
time 0.2100000000000001
time 0.2150000000000001
time 0.2200000000000001
time 0.22500000000000012
time 0.23000000000000012
time 

## 3D Pendulum Geometry with Rotation Axis

In [19]:
# Create two half circle for pendulum rod in 3D
rect1 = MoveTo(-1, -1).Rectangle(1, 2).Face()
rect2 = MoveTo(0, -1).Rectangle(1, 2).Face()

rect1 = rect1.Extrude(10)
rect2 = rect2.Extrude(10)

geo = Glue([rect1, rect2])
geo.edges[47].name = "fix"

x_axis=Axis((0,0,0), (1,0,0))
z_axis=Axis((0,0,0), (0,0,1))

geo = geo.Rotate(x_axis, 90)
geo = geo.Rotate(z_axis, -45)

DrawGeo(geo)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': …

BaseWebGuiScene

In [25]:
mesh = Mesh(OCCGeometry(geo, dim=3).GenerateMesh(maxh=0.5))
mesh.Curve(2)
Draw(mesh, view="show=all")

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [26]:
mesh.GetBBoundaries()

('default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'fix',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default',
 'default')

In [27]:
E, nu = 210e6, 0.2
rho = 7e3
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

I = Id(mesh.dim)

def C(u): 
    F = I+Grad(u)
    return F.trans*F
def NeoHooke (C):
    return 0.5 * mu * (Trace(C-I) + 2*mu/lam * Det(C)**(-lam/2/mu) - 1)

In [29]:
rho = 7e3
g = 9.81

fes = VectorH1(mesh, order=3, dirichlety="fix")
u,v = fes.TnT()

force = CF((0, -g*rho, 0))

uold = GridFunction(fes)
unew = GridFunction(fes)
vel = GridFunction(fes)
anew = GridFunction(fes)
aold = GridFunction(fes)

tau = 5e-3
tend = 1

bfmstar = BilinearForm(fes)
bfmstar += Variation( NeoHooke (C(u)).Compile(True)*dx )
bfmstar += Variation( -force*u*dx )
bfmstar += Variation( rho/2* 2/tau**2 * (u-uold-tau*vel-tau**2/4*aold)**2 * dx )

t = 0
unew.Set( (0,0,0) )
vel.Set( (0,0,0) )

X = CoefficientFunction((x,y,z))

if True:
    cf = (X + u-uold - (X.Other() + u.Other() - uold.Other())) * (-specialcf.normal(3).Other())
    # cf = (X + u-uold)*specialcf.normal(2) + \
    #    (X.Other() + u.Other() - uold.Other()) * specialcf.normal(2).Other()
    contact.AddEnergy(IfPos(cf, 1e9*cf*cf, 0), deformed=True)

In [30]:
scene = Draw (unew, mesh, "disp", deformation=unew)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [31]:
with TaskManager():
    while t < tend:
        print ("time", t)
        t += tau
        uold.vec.data = unew.vec
        aold.vec.data = anew.vec

        #contact.Update(uold, bfmstar, 5, 0.01)
        NewtonMinimization (a=bfmstar, u=unew, printing=False, inverse="sparsecholesky")

        anew.vec.data = unew.vec-uold.vec-tau*vel.vec-tau**2/4*aold.vec
        anew.vec.data *= 4/tau**2
        vel.vec.data += 0.5*tau*aold.vec
        vel.vec.data += 0.5*tau*anew.vec

        scene.Redraw()

time 0
time 0.005
time 0.01
time 0.015
time 0.02
time 0.025
time 0.030000000000000002
time 0.035
time 0.04
time 0.045
time 0.049999999999999996
time 0.05499999999999999
time 0.05999999999999999
time 0.06499999999999999
time 0.06999999999999999
time 0.075
time 0.08
time 0.085
time 0.09000000000000001
time 0.09500000000000001
time 0.10000000000000002
time 0.10500000000000002
time 0.11000000000000003


KeyboardInterrupt: 

## 3D Pendulum Head Impact on a Wall


In [ ]:
# Parameters Geometry Penudlum
l_rod = 0.400     # length of rod in m
r_rod = 0.0075   # radius of rod in m
r_head = 0.035   # radius of head in m
q0 = -45  # initial angle in rad

# Parameters Geometry Wall
wall_height = r_head * 4
wall_thickness = r_head / 2
wall_depth = r_head * 4

# Define Points
base_pt = (0, 0, 0)
top_pt = (0, l_rod, 0)


# Create Pendulum
rod = Cylinder(base_pt, Y, r_rod, l_rod)

head = Sphere(top_pt, r_head)
head.faces.name = "contact_head"




pendulum = rod + head
rot_axis = Axis(base_pt, Z)
pendulum = pendulum.Rotate(rot_axis, q0)

# Create Wall
wall_p1 = (-r_head-wall_thickness, -l_rod-wall_height/2, -wall_depth/2)
wall_p2 = (-r_head, -l_rod+wall_height/2, wall_depth/2)
wall = Box(wall_p1, wall_p2)

wall.faces.Max(X).name = "contact_wall"

wall.faces[5:9].name = "fixed_wall"


geo = Compound([pendulum, wall])

DrawGeo(geo)

TypeError: Max(): incompatible function arguments. The following argument types are supported:
    1. (self: netgen.libngpy._NgOCC.ListOfShapes, dir: netgen.libngpy._NgOCC.gp_Vec) -> object

Invoked with: <netgen.libngpy._NgOCC.ListOfShapes object at 0x76c51019ccf0>, <ngsolve.fem.CoefficientFunction object at 0x76c510553d70>

In [ ]:
geo = OCCGeometry(geo, dim=3)
mesh = Mesh(geo.GenerateMesh(maxh=0.02))
mesh.Curve(2)
Draw(mesh)